In [2]:
import pandas as pd
import numpy as np

In [59]:
import pandas as pd

df = pd.read_excel(r"C:\Users\m.olshanskiy\Desktop\Пыпин 2025 квартиры.xlsx")
df2 = pd.read_excel(r"C:\Users\m.olshanskiy\PycharmProjects\ndv_parsing\База данных\Выгрузка01-2025.xlsx")



C:\Users\m.olshanskiy\AppData\Local\Temp\ipykernel_13004\3669503103.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  .apply(pick_match)
C:\Users\m.olshanskiy\AppData\Local\Temp\ipykernel_13004\3669503103.py:43: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_match)


In [62]:
df = df.reset_index(drop=False).rename(columns={'index': 'deal_id'})

df['month'] = df['Дата ДДУ'].dt.to_period('M')
df2['month'] = df2['date'].dt.to_period('M')

merged = df.merge(
    df2,
    left_on=['month', 'ID ЖК'],
    right_on=['month', 'id'],
    how='left'
)

merged['area_diff'] = (merged['Площадь'] - merged['area_sqm']).abs()

def pick_match(group):
    valid = group[group['area_diff'] <= 0.2]

    if not valid.empty:
        # ближайшее по площади
        return valid.loc[[valid['area_diff'].idxmin()]]
    else:
        # сохраняем сделку без совпадения
        row = group.iloc[[0]].copy()
        row[df2.columns] = pd.NA
        row['area_diff'] = pd.NA
        return row


result = (
    merged
    .groupby('deal_id', as_index=False)
    .apply(pick_match)
    .reset_index(drop=True)
)

result = result.drop(columns=['id'])

print(len(df), len(result))


C:\Users\m.olshanskiy\AppData\Local\Temp\ipykernel_13004\977794494.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  .apply(pick_match)


27442 27442


C:\Users\m.olshanskiy\AppData\Local\Temp\ipykernel_13004\977794494.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_match)


In [63]:
result.to_excel(r"C:\Users\m.olshanskiy\Desktop\Пыпин результат.xlsx", index=False)